# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task.

## Packages and Data

In [1]:
# packages

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset
from src.fine_tuning.modeling import FineTuneLLM

In [13]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS
from src.config import GENERATION_MAX_LENGTH, GENERATION_NUM_BEAMS

## save model
from src.config import OUTPUT_DIR

In [3]:
# get data
sf = ScryfallDataset(task = TASK)

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 139
	Validation Records = 35
	Test Records = 5
	Records saved to...
		../data/scryfall_seq2seq_train.json
		../data/scryfall_seq2seq_val.json
		../data/scryfall_seq2seq_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Seq2Seq Dataset Loaded
	Train Records = 139
	Val Records = 35
	Test Records = 5
	Count Unique Tags = 0


## Modeling

In [4]:
# from transformers import Trainer
# from torch.nn import BCEWithLogitsLoss

# class CustomTrainer(Trainer):
#     def __init__(self, pos_weights, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.pos_weights = pos_weights

#     def compute_loss(self, model, inputs, return_outputs = False):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs) 
#         logits = outputs.logits

#         loss_fct = BCEWithLogitsLoss(pos_weight = self.pos_weights)
#         loss = loss_fct(logits, labels)

#         return (loss, outputs) if return_outputs else loss

In [5]:
# fine tune the model
tagger = FineTuneLLM(
    model_name = MODEL_NAME,
    dataset = sf.dataset
)
tagger.prepare_data(
    max_input_length = MAX_INPUT_LENGTH,
    max_target_length = MAX_TARGET_LENGTH
)
tagger.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    generation_max_length = GENERATION_MAX_LENGTH,
    generation_num_beams = GENERATION_NUM_BEAMS
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/139 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


In [6]:
import pandas as pd

def debug_model_outputs(tagger, raw_dataset, num_samples=5):
    """
    Grabs samples from the validation set and compares ground truth to model output.
    """
    print(f"\n--- Model Output Debugging ({num_samples} Samples) ---")
    print(f'[GT = Ground Truth, PR = Model Prediction]')
    
    # Access the original validation data before it was tokenized/stripped
    # We need the 'document' for input and 'tags' for comparison
    val_data = raw_dataset['val']
    
    results = []
    
    for i in range(min(num_samples, len(val_data))):
        sample = val_data[i]
        card_text = sample['document']
        ground_truth = sample['tags']
        
        # Generate prediction using the model's current state
        # generate_tags internally uses self.model.generate()
        predicted_tags = tagger.generate_tags(card_text)
        
        # Format for display
        results.append({
            "Card Text": card_text[:100] + "...", 
            "Ground Truth": ground_truth,
            "Model Prediction": ", ".join(list(predicted_tags)) if predicted_tags else "[EMPTY]"
        })
    
    # Display results
    df = pd.DataFrame(results)
    for idx, row in df.iterrows():
        print(f"\nSample {idx+1}:")
        print(f"  GT: {row['Ground Truth']}")
        print(f"  PR: {row['Model Prediction']}")

# Usage:
# Assuming 'tagger' is your FineTuneLLM instance
debug_model_outputs(tagger, sf.dataset, num_samples=10)


--- Model Output Debugging (10 Samples) ---
[GT = Ground Truth, PR = Model Prediction]

Sample 1:
  GT: brinebarrow intruder, cogwork wrestler, combat trick, shrink, triggered ability, virtual vanilla
  PR: power, toughness, target creature

Sample 2:
  GT: chromatic lantern, gives mana ability, harvest mage, mana fix, rain of filth
  PR: mana value, mana

Sample 3:
  GT: cheat death, death trigger, regrowth-creature
  PR: enchanted creature

Sample 4:
  GT: activated ability, bottomless mana sink, regenerates self, shade pump, type errata
  PR: stalker

Sample 5:
  GT: evasion, french vanilla
  PR: soul of the rapids mana cost

Sample 6:
  GT: cheaper than mv, gives unblockable, ninjutsu, noncreature typal, saboteur, synergy-attacker, triggered ability, tutor-creature-ninja, tutor-to-hand, typal-ninja
  PR: 5.0 type line, mana value, legendary creature — human ninja rules text, higure, ninjutsu, the still wind mana cost

Sample 7:
  GT: activated ability, armadillo cloak, cycle-arb-a

In [7]:
record = sf.dataset['test'][1]
pred_tags = tagger.generate_tags(card_text = record['document'])
print(f'Card\n{record["document"]}')
print(f'Actual Tags = {record["tags"]}')
print(f'Predicted Tags = {pred_tags}')

Card

        Generate comma-separated Scryfall community tags for the following card:

        ----------
        Airdrop Aeronauts
        Mana Cost = {3}{W}{W}
Mana Value = 5.0

        Type Line = Creature — Dwarf Scout

        Rules Text = Flying
Revolt — When this creature enters, if a permanent left the battlefield under your control this turn, you gain 5 life.
 
        Power = 4
Toughness = 3

        
        Color Identity = ['W']

        Rarity = uncommon
        ----------
        
Actual Tags = alliteration, evasion, intervening if clause, leaves battlefield trigger, lifegain, revolt, virtual french vanilla
Predicted Tags = {'dwarf'}


## Save To Huggingface Hub

In [16]:
# login to the hugging face
from huggingface_hub import notebook_login
with open('../huggingface_token.txt', 'r') as f:
    token = f.read()

notebook_login()

In [17]:
# upload the model to the huggingface hub
from huggingface_hub import Repository
from huggingface_hub import get_full_repo_name

## define the repo locally
## NOTE: Be sure to create OUTPUT_DIR in the hub manually first
repo_name = get_full_repo_name(OUTPUT_DIR)
repo = Repository(OUTPUT_DIR, clone_from = repo_name)

## save to hub
finetune.save_to_huggingface_hub(
    output_dir = OUTPUT_DIR,
    repo = repo,
    commit_message = f'Fine-tuned {MODEL} on scryfall tags.'
)

ImportError: cannot import name 'Repository' from 'huggingface_hub' (c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\huggingface_hub\__init__.py)

In [18]:
import huggingface_hub

In [ ]:
huggingface_hub.